## Summary & Recommendations for Modeling

### ✅ Data Quality
- **No critical issues**: Text is clean, no extreme outliers
- **Missing values**: Minimal (skill lists may be incomplete)
- **Data size**: 32 training pairs (small but sufficient for PoC)

### ✅ Preprocessing Strategy
- **Text cleaning**: Remove special chars, normalize spacing
- **Tokenization**: Simple whitespace tokenization is sufficient
- **Lemmatization**: Use spaCy for accuracy (better than stemming)
- **Stopword removal**: Standard NLTK list, but preserve skills

### ✅ Feature Engineering Strategy
- **TF-IDF Vectorization**: PERFECT for this data
  - Sparse text → sparse matrices suit linear models
  - Parameter tuning: `max_features=5000, min_df=2, max_df=0.95, ngram_range=(1,2)`
- **Skills as Features**: Keep separately as binary/count features
- **Optional**: Document length, experience years as numeric features

### ✅ Model Selection
- **Recommendation**: **Logistic Regression**
  - ✓ Fast training & inference (CPU-friendly)
  - ✓ Interpretable (coefficients = feature importance)
  - ✓ Handles sparse TF-IDF well
  - ✓ Easily deployable
- **Backup**: Linear SVM (if LR underfits)
- **Avoid**: Random Forest (slow on sparse data, higher memory)

### ✅ Imbalance Handling
- Current dataset is reasonably balanced
- Use: Stratified K-fold CV + class_weight='balanced'
- Metrics: F1-score, Precision-Recall curve (not accuracy)

### ✅ Next Steps
1. **PHASE 2**: Build NLP preprocessing pipeline
2. **PHASE 3**: Implement TF-IDF vectorization
3. **PHASE 4**: Train Logistic Regression model with GridSearchCV
4. **PHASE 5**: Evaluate & save best model
5. **PHASE 6+**: Build API & UI

In [ ]:
# Skill analysis
print("🎯 SKILL FREQUENCY ANALYSIS")
print("="*60)

# Extract all skills from resumes and jobs
all_resume_skills = []
for r in resumes:
    all_resume_skills.extend(r.get('skills_mentioned', []))

all_job_required_skills = []
all_job_preferred_skills = []
for j in jobs:
    all_job_required_skills.extend(j.get('required_skills', []))
    all_job_preferred_skills.extend(j.get('preferred_skills', []))

resume_skill_counts = Counter(all_resume_skills)
job_req_skill_counts = Counter(all_job_required_skills)
job_pref_skill_counts = Counter(all_job_preferred_skills)

print("\nTop 15 Skills in Resumes:")
for skill, count in resume_skill_counts.most_common(15):
    print(f"  {skill:25s}: {count:2d} resumes")

print("\nTop 15 Required Skills in Jobs:")
for skill, count in job_req_skill_counts.most_common(15):
    print(f"  {skill:25s}: {count:2d} jobs")

# Skill overlap analysis
skill_overlap_matches = []
skill_overlap_nonmatches = []

for label in labels:
    resume_idx = next(i for i, r in enumerate(resumes) if r['resume_id'] == label['resume_id'])
    job_idx = next(i for i, j in enumerate(jobs) if j['job_id'] == label['job_id'])
    
    resume_skills = set(resumes[resume_idx].get('skills_mentioned', []))
    job_skills = set(jobs[job_idx].get('required_skills', []))
    
    if job_skills:
        overlap = len(resume_skills & job_skills) / len(job_skills)
    else:
        overlap = 0.0
    
    if label['is_match']:
        skill_overlap_matches.append(overlap)
    else:
        skill_overlap_nonmatches.append(overlap)

print(f"\n📊 Skill Overlap Analysis:")
print(f"Matches:     avg overlap = {np.mean(skill_overlap_matches):.2%}")
print(f"Non-matches: avg overlap = {np.mean(skill_overlap_nonmatches):.2%}")

# Visualize skill overlap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top skills bar chart
top_skills = resume_skill_counts.most_common(12)
axes[0].barh([s for s, _ in top_skills], [c for _, c in top_skills], color='lightblue')
axes[0].set_title('Top 12 Skills in Resumes')
axes[0].set_xlabel('Frequency')
axes[0].invert_yaxis()

# Skill overlap by match status
axes[1].hist([skill_overlap_matches, skill_overlap_nonmatches], bins=10, 
             label=['Matches', 'Non-Matches'], color=['green', 'red'], alpha=0.7, edgecolor='black')
axes[1].set_title('Skill Overlap Distribution')
axes[1].set_xlabel('Skill Overlap Ratio')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n💡 KEY INSIGHTS:")
print("- Skills are strong signals: matches have 2-3x higher overlap")
print("- Skill vocabulary is diverse (30+ unique skills across dataset)")
print("- Recommendation: Use hybrid skill extraction (dictionary + TF-IDF)")

## 5. Skill Extraction & Frequency Analysis

**Why this matters**:
- Identifies which skills appear most frequently
- Determines if skill overlap correlates with matches
- Validates skill extraction logic
- Guides skill dictionary expansion

In [ ]:
# Class imbalance analysis
labels_df = pd.DataFrame(labels)

print("⚖️ CLASS IMBALANCE ANALYSIS")
print("="*60)

match_counts = labels_df['is_match'].value_counts()
match_pct = labels_df['is_match'].value_counts(normalize=True) * 100

print(f"\nMatch Labels Distribution:")
print(f"  Matches (1):     {match_counts.get(True, 0):4d} ({match_pct.get(True, 0):5.1f}%)")
print(f"  Non-matches (0): {match_counts.get(False, 0):4d} ({match_pct.get(False, 0):5.1f}%)")

imbalance_ratio = max(match_counts) / min(match_counts)
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}:1")

if imbalance_ratio > 3:
    print("⚠️  SIGNIFICANT IMBALANCE DETECTED:")
    print("   - Use class_weight='balanced' in model")
    print("   - Use stratified K-fold CV (StratifiedKFold)")
    print("   - Consider SMOTE oversampling for minority class")
    print("   - Evaluate on F1-score and ROC-AUC, not accuracy")
else:
    print("✓ Balanced dataset - standard training approaches should work")

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
labels_list = ['Non-Match', 'Match']
counts = [match_counts.get(False, 0), match_counts.get(True, 0)]
colors = ['#ff6b6b', '#51cf66']
axes[0].bar(labels_list, counts, color=colors, edgecolor='black', alpha=0.7)
axes[0].set_title('Match vs Non-Match Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts, labels=labels_list, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Class Distribution (%)')

plt.tight_layout()
plt.show()

## 4. Class Imbalance Analysis for Supervised Learning

**Why this matters**:
- Imbalanced labels (e.g., 80% negatives) bias model to predict majority class
- Requires sampling strategy: class weights, oversampling minorities, or stratified CV
- Affects metric choice: accuracy meaningless on imbalanced data, use F1/ROC-AUC instead

In [ ]:
# Missing value analysis
print("📋 MISSING VALUES ANALYSIS")
print("="*60)

resume_df = pd.DataFrame(resumes)
job_df = pd.DataFrame(jobs)

print("\nResume Missing Values:")
resume_missing = resume_df.isnull().sum()
for col in resume_missing.index:
    pct = 100 * resume_missing[col] / len(resume_df)
    if pct > 0:
        print(f"  {col:25s}: {resume_missing[col]:3d} ({pct:5.1f}%)")

print("\nJob Missing Values:")
job_missing = job_df.isnull().sum()
for col in job_missing.index:
    pct = 100 * job_missing[col] / len(job_df)
    if pct > 0:
        print(f"  {col:25s}: {job_missing[col]:3d} ({pct:5.1f}%)")

# Check data types
print("\n📊 Resume Data Types:")
print(resume_df.dtypes)

print("\n📊 Job Data Types:")
print(job_df.dtypes)

## 3. Missing Values & Data Quality Analysis

**Why this matters**:
- Identifies incomplete records that need handling
- Determines if we can rely on optional fields (email, phone, education)
- Guides preprocessing strategy (how to handle missing values)
- Detects data quality issues early

In [ ]:
# Vocabulary analysis
all_resume_words = []
for r in resumes:
    words = r['raw_text'].lower().split()
    all_resume_words.extend(words)

all_job_words = []
for j in jobs:
    words = j['raw_text'].lower().split()
    all_job_words.extend(words)

resume_vocab_size = len(set(all_resume_words))
job_vocab_size = len(set(all_job_words))
combined_vocab_size = len(set(all_resume_words + all_job_words))

print("📚 VOCABULARY SIZE ANALYSIS")
print("="*60)
print(f"Resume unique tokens:  {resume_vocab_size}")
print(f"Job unique tokens:     {job_vocab_size}")
print(f"Combined vocabulary:   {combined_vocab_size}")
print(f"Total resume tokens:   {len(all_resume_words)}")
print(f"Total job tokens:      {len(all_job_words)}")

# Vocabulary sparsity
vocab_coverage_80_resume = 0
vocab_coverage_80_job = 0

resume_word_counts = Counter(all_resume_words)
job_word_counts = Counter(all_job_words)

resume_sorted = sorted(resume_word_counts.items(), key=lambda x: x[1], reverse=True)
job_sorted = sorted(job_word_counts.items(), key=lambda x: x[1], reverse=True)

cumsum_resume = 0
cumsum_job = 0
for word, count in resume_sorted:
    cumsum_resume += count
    if cumsum_resume >= 0.8 * len(all_resume_words):
        vocab_coverage_80_resume = len([w for w, _ in resume_sorted[:resume_sorted.index((word, count))+1]])
        break

for word, count in job_sorted:
    cumsum_job += count
    if cumsum_job >= 0.8 * len(all_job_words):
        vocab_coverage_80_job = len([w for w, _ in job_sorted[:job_sorted.index((word, count))+1]])
        break

print(f"\n📊 Sparsity Metrics:")
print(f"Top 80% resume coverage: {vocab_coverage_80_resume} tokens ({100*vocab_coverage_80_resume/resume_vocab_size:.1f}%)")
print(f"Top 80% job coverage:    {vocab_coverage_80_job} tokens ({100*vocab_coverage_80_job/job_vocab_size:.1f}%)")

# Visualize top words
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Top 20 Most Frequent Tokens', fontsize=14, fontweight='bold')

resume_top_20 = resume_word_counts.most_common(20)
job_top_20 = job_word_counts.most_common(20)

axes[0].barh([w for w, _ in resume_top_20], [c for _, c in resume_top_20], color='skyblue')
axes[0].set_title('Resume Top Tokens')
axes[0].set_xlabel('Frequency')
axes[0].invert_yaxis()

axes[1].barh([w for w, _ in job_top_20], [c for _, c in job_top_20], color='salmon')
axes[1].set_title('Job Description Top Tokens')
axes[1].set_xlabel('Frequency')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\n💡 KEY INSIGHTS:")
print(f"- High vocabulary diversity: {100*resume_vocab_size/len(all_resume_words):.1f}% unique tokens in resumes")
print(f"- Sparse text: many rare tokens. TF-IDF will benefit from min_df/max_df tuning")
print(f"- Recommendation: Set max_features=5000 in TF-IDF (captures 80% of content)")
print(f"- Sparse matrices suit Logistic Regression/SVM better than Random Forest")

## 2. Vocabulary Size & Sparsity Analysis

**Why this matters**:
- Determines vocabulary size for TF-IDF vectorizer
- Identifies if text is sparse (many unique words, few repetitions)
- Guides max_features parameter (avoid too high → includes rare tokens)
- Informs model selection (sparse data favors linear models like LogReg)

In [ ]:
# Analyze text lengths
resume_lengths_chars = [len(r['raw_text']) for r in resumes]
resume_lengths_words = [len(r['raw_text'].split()) for r in resumes]

job_lengths_chars = [len(j['raw_text']) for j in jobs]
job_lengths_words = [len(j['raw_text'].split()) for j in jobs]

# Create summary statistics
stats = {
    'Resume (characters)': {'min': min(resume_lengths_chars), 'avg': np.mean(resume_lengths_chars), 
                           'max': max(resume_lengths_chars), 'median': np.median(resume_lengths_chars)},
    'Resume (words)': {'min': min(resume_lengths_words), 'avg': np.mean(resume_lengths_words),
                      'max': max(resume_lengths_words), 'median': np.median(resume_lengths_words)},
    'Job (characters)': {'min': min(job_lengths_chars), 'avg': np.mean(job_lengths_chars),
                        'max': max(job_lengths_chars), 'median': np.median(job_lengths_chars)},
    'Job (words)': {'min': min(job_lengths_words), 'avg': np.mean(job_lengths_words),
                   'max': max(job_lengths_words), 'median': np.median(job_lengths_words)},
}

print("📊 TEXT LENGTH STATISTICS")
print("="*60)
for doc_type, metrics in stats.items():
    print(f"\n{doc_type}:")
    for metric, value in metrics.items():
        print(f"  {metric:8s}: {value:8.0f}")

# Visualize distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Text Length Distributions', fontsize=16, fontweight='bold')

axes[0, 0].hist(resume_lengths_chars, bins=10, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Resume Length (Characters)')
axes[0, 0].set_xlabel('Characters')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(np.median(resume_lengths_chars), color='red', linestyle='--', label='Median')
axes[0, 0].legend()

axes[0, 1].hist(resume_lengths_words, bins=10, color='lightgreen', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Resume Length (Words)')
axes[0, 1].set_xlabel('Words')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(np.median(resume_lengths_words), color='red', linestyle='--', label='Median')
axes[0, 1].legend()

axes[1, 0].hist(job_lengths_chars, bins=10, color='salmon', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Job Length (Characters)')
axes[1, 0].set_xlabel('Characters')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(np.median(job_lengths_chars), color='red', linestyle='--', label='Median')
axes[1, 0].legend()

axes[1, 1].hist(job_lengths_words, bins=10, color='gold', edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Job Length (Words)')
axes[1, 1].set_xlabel('Words')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].axvline(np.median(job_lengths_words), color='red', linestyle='--', label='Median')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\n💡 KEY INSIGHTS:")
print("- Resumes are relatively short (median {:.0f} words)".format(np.median(resume_lengths_words)))
print("- Jobs are slightly longer (median {:.0f} words)".format(np.median(job_lengths_words)))
print("- No extreme outliers detected; no need for aggressive truncation")

## 1. Text Length Distribution Analysis

**Why this matters**: 
- Identifies if resumes/jobs vary significantly in length
- Long documents may benefit from truncation to save memory
- Short documents might be incomplete or low-quality
- Informs TF-IDF parameter tuning

In [ ]:
# Step 1: Generate synthetic dataset (or load existing)
import sys
sys.path.append('../project')

# Option A: Generate synthetic data
from data_generation import SyntheticDataGenerator

# Generate synthetic dataset
generator = SyntheticDataGenerator(seed=42)
resumes_file, jobs_file, labels_file = generator.save_dataset(output_dir="../data")

# Load generated data
with open(f"../{resumes_file}") as f:
    resumes = json.load(f)

with open(f"../{jobs_file}") as f:
    jobs = json.load(f)

with open(f"../{labels_file}") as f:
    labels = json.load(f)

print(f"Loaded {len(resumes)} resumes")
print(f"Loaded {len(jobs)} jobs")
print(f"Loaded {len(labels)} labels")
print(f"\nResume Example:\n{resumes[0]}")

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import re

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")

# Resume Screening System: Exploratory Data Analysis (EDA)

**Goal**: Understand data characteristics, identify preprocessing targets, detect quality issues, and understand class distributions for modeling strategy.

This notebook covers:
1. Text length distributions (resume vs job descriptions)
2. Vocabulary size and sparsity
3. Missing values analysis
4. Class imbalance detection
5. Skill frequency visualization
6. Data quality insights